# Apt 305 — canonical trajectory, green suite, and the wind diagnostic

Three things, runnable independently. **Run section 1 first** (setup); after that
each of 2, 3 and 4 stands alone.

| § | What | Runtime |
| --- | --- | --- |
| 2 | **Regression suite** — it used to exit 2, a *collection* error, so no assertion ran | ~5 min |
| 3 | **Canonical trajectory in methodology order** — literature corrections first, then the found defects, then the closure fixes | ~6 min |
| 4 | **Wind diagnostic** — why the wind-dependent `h_ce` moved sensible cooling | ~1 min |

**The canonical result does not change here**: 122.69 kWh sensible heating +
67.12 kWh sensible cooling + 3.98 kWh gated latent = 193.79 kWh =
**9.69 kWh/m²·yr**, V2 residual gate passing on every state.

Section 4 returns **verdict (c)** and escalates: four months of the weather
file's wind column are identically zero, and 96 % of the `h_ce` cooling increase
comes from exactly those hours. Read it before defending that correction.

CPU runtime is fine throughout.

## 1 · Setup  *(run this first)*

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO   = Path('/content/AIB')
BRANCH = 'claude/aib-energy-balance-closure-83epu7'
EPW    = 'weather_cache/AUS_VIC_Melbourne.RO.948680_TMYx.2011-2025.epw'

if not REPO.exists():
    subprocess.run(['git', 'clone',
                    'https://github.com/samiraghafarigousheh-sys/aib.git', str(REPO)],
                   check=True)
os.chdir(REPO)

# Every branch, not just the default one: section 3 builds one worktree per state
# and each state is a commit reachable only through those branches.
subprocess.run(['git', 'fetch', 'origin', '+refs/heads/*:refs/remotes/origin/*'], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)

# Colab already ships numpy, pandas, matplotlib, plotly, scipy, scikit-learn,
# pytz, requests, tqdm and pytest. These five are the gaps.
#
# pyecharts is only a rendering library and the engine no longer needs it to
# import -- that was the collection error section 2 fixes -- but the suite drives
# worktrees of the HISTORICAL branches, whose package still imports it eagerly.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pvlib', 'timezonefinder', 'holidays', 'workalendar', 'pyecharts'],
               check=True)

print(subprocess.run(['git', 'log', '--oneline', '-4'], capture_output=True, text=True).stdout)


def run(cmd, **kw):
    # Stream a subprocess, rather than going quiet for several minutes.
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, **kw)
    for line in p.stdout:
        print(line, end='')
    p.wait()
    return p.returncode

## 2 · The regression suite

It used to exit **2**. That is a *collection* error, not a test failure (exit 1)
— nothing ran, and a suite that cannot load is easy to mistake for one that
passes if only the exit code is read. Two causes, both import resolution:

* `pybuildingenergy/__init__.py` eagerly imported the charting layer, so the
  engine could not be imported at all without `pyecharts`;
* the repository root holds a directory named `pybuildingenergy/` with no
  `__init__.py`, which under PEP 420 is a valid **namespace package** of the same
  name — with no `source` submodule. It shadowed the real one, and the broken
  resolution then cached in `sys.modules` and took every later module with it.

Both are fixed. No assertion was weakened, skipped or removed.

In [ ]:
print('=== the three files named in the plan ===')
rc3 = run([sys.executable, '-m', 'pytest',
           'tests/test_sankey_closure_adj_transmission.py',
           'tests/test_latent_gating.py',
           'tests/test_gr_classification.py',
           '-v', '--no-header', '-p', 'no:cacheprovider'])
print(f'\nexit code: {rc3}   (0 = all assertions ran and passed, '
      f'1 = a real failure, 2 = collection error)')

In [ ]:
# The whole suite, from a clean checkout with no PYTHONPATH set -- which is the
# case the conftest fix exists for.
rc_all = run([sys.executable, '-m', 'pytest', 'tests/', '-q', '-p', 'no:cacheprovider'])
print(f'\nexit code: {rc_all}')
assert rc_all == 0, 'the suite is not green -- do not treat anything below as regression-tested'
print(open('results/tests/pytest_green.txt').read()[:1200])

## 3 · The canonical trajectory, in methodology order

The previous trajectory took each state from a branch tip, and every one of those
branches was cut from the unmodified baseline — so **neither literature
correction was in it**:

```
git merge-base --is-ancestor a66eec7 <each of the six state branches>  ->  no   (C1)
git merge-base --is-ancestor 56f5d08 <each of the six state branches>  ->  no   (C2)
```

C1 and C2 only appeared at the end, because `main` happened to carry them, which
made C2's ordinal meaningless. This rebuilds the stack from the baseline with
literature first:

```
Baseline -> C1 -> C2 -> ventilation -> latent -> internal gains
         -> conditioned zones -> ground -> hemisphere -> closure
```

Ventilation and latent are **split** (they were already two commits, so it was
free), and C1 **is** included as a cumulative step.

The harness refuses to guess: a cherry-pick conflict anywhere under
`pybuildingenergy/src/` aborts the run, because that is exactly the signal that a
correction is not cleanly separable at that point in the order.

In [ ]:
rc = run([sys.executable, 'tools/diagnostics/canonical_trajectory.py',
          '--weather', EPW, '--outdir', 'results/au_canonical'])
print(f'\nexit code: {rc}   '
      f'(0 = gate passed, HEAD invariant, final engine identical to HEAD; '
      f'2 = a state failed the gate; 3 = the canonical figure moved; '
      f'4 = the final engine differs from HEAD)')

In [ ]:
import json
import pandas as pd

blob = json.loads(Path('results/au_canonical/trajectory_raw.json').read_text())
res  = blob['results']

traj = pd.DataFrame([{
    'State': s,
    'Sensible heating': r['config_B']['Q_H_sensible_kWh'],
    'Sensible cooling': r['config_B']['Q_C_sensible_kWh'],
    'Latent (gated)':   r['config_B']['Q_C_latent_kWh'],
    'Latent (ungated)': r['config_B']['Q_C_latent_ungated_kWh'],
    'Total kWh':        r['config_B']['Q_need_total_kWh'],
    'Total kWh/m²':     r['config_B']['Q_need_total_kWh_per_sqm'],
    'Residual %':       r['config_B']['sankey']['residual_pct'],
    'Gate':             'PASS' if abs(r['config_B']['sankey']['residual_pct']) < 5 else 'FAIL',
    'Tr items':         r['config_B']['sankey']['n_transmission_items'],
} for s, r in res.items()])
display(traj.round(2))

assert (traj['Gate'] == 'PASS').all(), 'a state failed the 5 % gate'
print('Gate passed on all', len(traj), 'states.')

### Order-independence

Applying the same set of corrections in a different order must land on the same
engine. The check is made on the **source**, not only on the numbers — a
difference that happened not to move this particular building's annual result
would still be caught.

In [ ]:
chk  = blob['engine_tree_check']
canon = blob['canonical_check']

print('final engine tree vs HEAD:',
      'BYTE-FOR-BYTE IDENTICAL' if chk['identical'] else f"DIFFERS -> {chk['differing_files']}")
print()
for k, expected in canon['expected'].items():
    got = canon['final'][k]
    print(f"  {k:<28} expected {expected:8.2f}   got {got:8.2f}   Δ {abs(got-expected):.4f}")
print()
print('HEAD invariant under the reordering:', canon['ok'])
assert chk['identical'] and canon['ok'], 'the reordering moved the canonical result -- stop and report'

In [ ]:
rc = run([sys.executable, 'tools/diagnostics/make_closed_balance_chart.py',
          '--raw', 'results/au_canonical/trajectory_raw.json',
          '--outdir', 'results/au_canonical',
          '--stem', 'au_canonical_trajectory',
          '--canonical-state', '+Closure fixes',
          '--title', 'Apt 305, 50 Barry St Carlton — canonical trajectory, in methodology order',
          '--note', 'Engine: ISO 52016-1 with all corrections, including the wind-dependent '
                    'external h_ce = 4v + 4 (C2), applied at its methodology position rather '
                    'than incidentally at the end. Literature corrections (C1, C2) first, then '
                    'the implementation defects found, then the closure fixes. The final state, '
                    '+Closure fixes, is CANONICAL and is byte-for-byte the HEAD engine. Every '
                    'state measured with the same reporting instrument; each metric on its own axis.'])

from IPython.display import Image, Markdown, display
display(Image('results/au_canonical/au_canonical_trajectory.png'))

In [ ]:
display(Markdown(Path('results/au_canonical/comparison.md').read_text()))

## 4 · Wind diagnostic — why `h_ce` moved sensible cooling

C2 replaces the ISO fixed external convective coefficient with `h_ce = 4v + 4`.
The pivot is **v = 4 m/s**, where `4v + 4 = 20 W/(m²·K)` — exactly the fixed
value. Above it the surface is coupled more tightly to outdoor air; below it,
less.

The plan's three candidate readings:

* **(a)** many hours above the pivot → more coupling year-round;
* **(b)** high winds coincide with hot cooling hours → cooling-season amplified;
* **(c)** neither — escalate, don't paper over.

This runs the **same engine twice**, changing only the h_ce model
(`external_convection_model='table'` recovers the ISO constant exactly), and
attributes the resulting change in cooling to bands of wind speed. That is what
actually decides between the three: if the high-wind branch drove it, the energy
would sit above 4 m/s.

In [ ]:
rc = run([sys.executable, 'tools/diagnostics/wind_h_ce_diagnostic.py',
          '--weather', EPW, '--outdir', 'results/diagnostics'])

from IPython.display import Image, Markdown, display
display(Image('results/diagnostics/wind_distribution.png'))

In [ ]:
import json
w = json.loads(Path('results/diagnostics/wind_stats.json').read_text())
s = w['summary']

print(f"VERDICT: ({w['verdict']})\n")
print(f"  hours above the 4 m/s pivot          {s['pct_hours_above_pivot']:6.1f} %")
print(f"  mean wind speed, whole year          {s['mean_wind_annual']:6.2f} m/s")
print(f"  mean wind, cooling-plant-on hours    {s['mean_wind_cooling_dyn']:6.2f} m/s"
      f"   ({s['wind_ratio_cooling_to_annual']:.2f}x annual -- CALMER, not windier)")
print()
print(f"  cooling, ISO fixed h_ce = 20         {s['C_fix']:6.2f} kWh   ({s['n_cooling_fix']} plant hours)")
print(f"  cooling, dynamic h_ce = 4v + 4       {s['C_dyn']:6.2f} kWh   ({s['n_cooling_dyn']} plant hours)")
print(f"  change                               {s['delta_C']:+6.2f} kWh")
print()
print("  where that change comes from:")
for b in s['bands']:
    print(f"    {b['label']:<12} {b['hours']:>5} h   {b['extra_cooling_kWh']:+8.2f} kWh   {b['share_pct']:6.1f} %")

import calendar
if s['degenerate_months']:
    names = ', '.join(calendar.month_abbr[m] for m in s['degenerate_months'])
    print(f"\n  *** DEGENERATE WIND MONTHS: {names} "
          f"({s['degenerate_hours']:,} h at exactly 0.0 m/s, "
          f"{100*s['degenerate_hours']/s['n_hours']:.1f} % of the year) ***")
    print(f"      the other months average {s['mean_wind_live_months']:.2f} m/s")

In [ ]:
display(Markdown(Path('results/diagnostics/wind_verdict.md').read_text()))

## 5 · Before quoting any of this

1. **The canonical figure is unchanged and was verified, not assumed.** The
   reordered trajectory's final state is byte-for-byte the HEAD engine, so
   `9.69 kWh/m²·yr` cannot have moved — and the notebook asserts it.

2. **`+Closure fixes` moves no number, and that is the finding.** The reporting
   instrument is held constant across all ten states — which is what makes them
   comparable — so the closure fixes' content is already in it. They correct the
   measurement, not the physics.

3. **The sensible-cooling column carries a caveat from §4.** Four months of the
   EPW's wind column are identically zero, two of them the peak cooling months,
   and 96 % of the `h_ce` cooling increase comes from exactly-zero-wind hours.
   Do not present `20.06 → 67.12 kWh` as a physical finding about wind-dependent
   convection until that column is re-sourced. With the ISO fixed coefficient the
   same engine gives 18.40 kWh.

4. **Quote the absolute change, not the ratio.** The "3×" is a small-denominator
   artefact: at C2's methodology position the same commit moves cooling
   749.71 → 869.29 kWh, +16 %.

5. **The six historical states in §3 are measured with a back-ported
   instrument.** The physics of each state is untouched; the mechanics are in the
   harness docstring and the report's provenance section.